In [1]:
pip install thefuzz python-Levenshtein

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from thefuzz import process, fuzz

cleaned_df = pd.read_csv('Unicorn_Startups_Cleaned.csv')
global_df = pd.read_csv('Unicorn_Companies.csv')

#global dataset for India only
india_global = global_df[global_df['Country'] == 'India'].copy()

print(f'Cleaned dataset: {cleaned_df.shape}')
print(f'Indian unicorns in global dataset: {india_global.shape}')

Cleaned dataset: (99, 14)
Indian unicorns in global dataset: (63, 13)


In [2]:
#Getting exact matches first

# Normalizing  names for matching
cleaned_df['name_lower'] = cleaned_df['startup_name'].str.strip().str.lower()
india_global['name_lower'] = india_global['Company'].str.strip().str.lower()

# Exact merge first
exact_merged = cleaned_df.merge(
    india_global[['name_lower', 'City', 'Select Inverstors', 'Total Raised', 'Investors Count']],
    on='name_lower',
    how='left'
)

exact_matches = exact_merged['City'].notna().sum()
print(f'Exact matches found: {exact_matches}')
print(f'Still unmatched: {exact_merged["City"].isna().sum()}')

Exact matches found: 46
Still unmatched: 53


In [4]:
#Fuzzy matching the remaining unmatched rows
#Getting the unmatched rows
unmatched_mask = exact_merged['City'].isna()
unmatched_names = exact_merged.loc[unmatched_mask, 'name_lower'].tolist()
global_names = india_global['name_lower'].tolist()

#Fuzzy matching each unmatched name
fuzzy_results = []
for name in unmatched_names:
    match, score = process.extractOne(name, global_names, scorer=fuzz.token_sort_ratio)
    fuzzy_results.append({
        'name_lower': name,
        'fuzzy_match': match,
        'score': score
    })

fuzzy_df = pd.DataFrame(fuzzy_results)
print(fuzzy_df.sort_values('score', ascending=False).to_string())

                          name_lower                   fuzzy_match  score
46                      ola electric         ola electric mobility     73
25                          cult.fit                       curefit     67
37                   glance (inmobi)                        inmobi     63
9                        leadsquared                     dealshare     60
15                           fractal             fractal analytics     58
26                             droom                     oyo rooms     57
45                           icertis                       licious     57
29                        innovaccer                      livspace     56
27                         pharmeasy                     sharechat     56
24              blinkit (ex-grofers)                       blinkit     56
32                              zeta                       zetwerk     55
2                              zepto                        upstox     55
7                            onecard  

In [5]:
#Reviewing and applying only high-confidence matches
#Only accepting matches with score >= 65
good_matches = fuzzy_df[fuzzy_df['score'] >= 65].copy()
print(f'High confidence fuzzy matches (score >= 65): {len(good_matches)}')
print(f'Low confidence (will be left as NaN): {len(fuzzy_df) - len(good_matches)}')

#Mapping fuzzy matches back to global dataset
fuzzy_lookup = india_global.set_index('name_lower')[
    ['City', 'Select Inverstors', 'Total Raised', 'Investors Count']
]

for _, row in good_matches.iterrows():
    idx = exact_merged[exact_merged['name_lower'] == row['name_lower']].index
    if row['fuzzy_match'] in fuzzy_lookup.index:
        for col in ['City', 'Select Inverstors', 'Total Raised', 'Investors Count']:
            exact_merged.loc[idx, col] = fuzzy_lookup.loc[row['fuzzy_match'], col]

print(f'\n After fuzzy matching, filled: {exact_merged["City"].notna().sum()} rows')
print(f'Still unmatched (newer startups): {exact_merged["City"].isna().sum()}')

High confidence fuzzy matches (score >= 65): 2
Low confidence (will be left as NaN): 51

 After fuzzy matching, filled: 48 rows
Still unmatched (newer startups): 51


In [6]:
#Cleaning up and renaming enriched columns

#Renaming columns cleanly
exact_merged = exact_merged.rename(columns={
    'Select Inverstors': 'investors',
    'Total Raised': 'total_raised',
    'Investors Count': 'investors_count',
    'City': 'city'
})

#Dropping the helper column
exact_merged = exact_merged.drop(columns=['name_lower'])

#Cleaning city names (Bengaluru vs Bangalore)
exact_merged['city'] = exact_merged['city'].replace({
    'Bangalore': 'Bengaluru',
    'Andheri': 'Mumbai',
    'Thane': 'Mumbai',
    'Maharashtra': 'Mumbai',
    'Gurugram': 'Gurgaon',
    'Faridabad': 'New Delhi'
})

print('City distribution:')
print(exact_merged['city'].value_counts())

City distribution:
city
Bengaluru    23
Gurgaon       9
Mumbai        5
Pune          4
New Delhi     3
Hyderabad     1
Jaipur        1
Chennai       1
Noida         1
Name: count, dtype: int64


In [7]:
#Cleaning Total Raised column

def clean_total_raised(val):
    if pd.isna(val):
        return np.nan
    val = str(val).replace('$', '').replace(',', '').strip()
    if 'B' in val:
        return float(val.replace('B', '').strip()) * 1000
    elif 'M' in val:
        return float(val.replace('M', '').strip())
    else:
        return np.nan

exact_merged['total_raised_million_usd'] = exact_merged['total_raised'].apply(clean_total_raised)
print(exact_merged[['startup_name', 'total_raised', 'total_raised_million_usd']].dropna().head(10))

   startup_name total_raised  total_raised_million_usd
10   ElasticRun     $432.11M                    432.11
13    DealShare     $391.52M                    391.52
14   Xpressbees     $472.94M                    472.94
19    Darwinbox      $106.7M                    106.70
20     Livspace     $371.69M                    371.69
22       Hasura     $136.51M                    136.51
27     NoBroker      $371.4M                    371.40
28     MobiKwik     $248.96M                    248.96
29       Spinny      $515.9M                    515.90
32    ShareChat      $1.277B                   1277.00


In [8]:
print("=== ENRICHED DATASET SUMMARY ===")
print(f"Shape: {exact_merged.shape}")
print(f"\nColumns: {exact_merged.columns.tolist()}")
print(f"\nMissing values:\n{exact_merged.isnull().sum()}")
print(f"\nCity coverage: {exact_merged['city'].notna().sum()}/99 startups")
print(f"Investor coverage: {exact_merged['investors'].notna().sum()}/99 startups")
print(f"Funding coverage: {exact_merged['total_raised_million_usd'].notna().sum()}/99 startups")

# Save
exact_merged.to_csv('Unicorn_Startups_Enriched.csv', index=False)
print("\n Enriched dataset saved!")

from IPython.display import FileLink
FileLink('Unicorn_Startups_Enriched.csv')

=== ENRICHED DATASET SUMMARY ===
Shape: (99, 19)

Columns: ['startup_name', 'industry', 'founding_year', 'unicorn_entry_year', 'profit_loss_fy22', 'current_valuation', 'acquisitions', 'status', 'valuation_million_usd', 'profit_loss_million_usd', 'industry_clean', 'status_clean', 'years_to_unicorn', 'is_profitable', 'city', 'investors', 'total_raised', 'investors_count', 'total_raised_million_usd']

Missing values:
startup_name                 0
industry                     0
founding_year                0
unicorn_entry_year           0
profit_loss_fy22            27
current_valuation            1
acquisitions                 6
status                       0
valuation_million_usd        1
profit_loss_million_usd     28
industry_clean               0
status_clean                 0
years_to_unicorn             0
is_profitable                0
city                        51
investors                   51
total_raised                51
investors_count             51
total_raised_million_usd

/Users/mehreenmuzammil/anaconda_projects/5c518ece-53de-4a07-a7b7-8e4997762bc6/Unicorn_Startups_Enriched.csv

In [9]:
import pandas as pd
df = pd.read_csv('Unicorn_Startups_Enriched.csv')
print(df.shape)
print(df.columns.tolist())
print(f"\nCity coverage: {df['city'].notna().sum()}/99")
print(f"Investor coverage: {df['investors'].notna().sum()}/99")
print(f"Funding coverage: {df['total_raised_million_usd'].notna().sum()}/99")

(99, 19)
['startup_name', 'industry', 'founding_year', 'unicorn_entry_year', 'profit_loss_fy22', 'current_valuation', 'acquisitions', 'status', 'valuation_million_usd', 'profit_loss_million_usd', 'industry_clean', 'status_clean', 'years_to_unicorn', 'is_profitable', 'city', 'investors', 'total_raised', 'investors_count', 'total_raised_million_usd']

City coverage: 48/99
Investor coverage: 48/99
Funding coverage: 48/99
